# Day 4.6 — Comparative Evaluation


## Before you begin

### Learning outcomes

- Score all three architectures on the same artifact with the same reviewer.
- Run the comparison twice, against two different reviewers, and read who won.
- State the conditions under which a multi-agent team is not worth building.

Architecture reference: [Day 4 diagrams D15](../../diagrams/source/day_04.md)

### Expected observation

Two tables. In the first the specialist team wins 9/9 against 5/9. In the second every system finds 8/9 and the team simply costs three times as much.


## Concept briefing

## Evaluating nondeterministic systems

Do not assert exact model wording. Assert invariants and measure outcomes:

- Is every finding structurally valid?
- Does the evidence quote the supplied artifact?
- How many known defects were found, counted once each?
- How many unsupported findings (false positives) were reported?
- How many duplicates survived synthesis, and how many did the supervisor merge?
- How many calls and tokens were used, according to the provider?
- Did the system terminate within its bounds, and what did it truncate?

Telemetry must be measured, never invented. A step that makes no model call reports zero
tokens and says "no model call"; it does not borrow a plausible-looking estimate. A tidy
number in a results table that nothing actually produced is worse than no number.

One run is an anecdote. Repeat model experiments with the same model, prompt version,
temperature and artifact, and report the variance rather than the best result.

## Capability can change the architecture conclusion

A weaker instruction-following model can benefit a lot from narrow prompts. A stronger
model may handle the general review well enough that the specialist calls add nothing.
So "multi-agent is better" usually means "decomposition compensated for *this* reviewer on
*this* task with *this* prompt."

Our two scenarios make that explicit and measurable rather than rhetorical. In the
`blind_spots` scenario the team finds 9/9 where one reviewer finds 5/9, and the extra
calls are clearly worth it. In the `strong_generalist` scenario the team finds exactly
what one reviewer already found, for three times the calls and tokens, plus duplicates to
merge and three times the surface area to debug. Same code, same artifact, opposite
verdict.

Multi-agent is not worth it when: one reviewer already reaches the quality bar; the extra
recall costs more than the defects it catches; the sub-tasks are not genuinely
independent; or the failure you keep hitting is a capability gap that a narrower prompt
cannot fill.

## Cost and latency arithmetic

Approximate run cost as:

```text
sum of input tokens across calls
+ sum of output and reasoning tokens
+ retries
```

Send the same 1,000-token artifact to three specialists and you pay for that input three
times, unless caching or a provider feature changes the arithmetic. Parallel execution can
cut elapsed time while leaving total cost identical or higher.

A fair comparison records recall, false positives, duplicates, calls, tokens, latency,
estimated cost and debugging burden. Then choose the **smallest** system that meets the
quality requirement.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — One function that scores all three systems

The reviewer is a parameter, the architectures are fixed. Changing one thing at a time is what makes this a comparison rather than a story.


In [ ]:
from review_team import (FallbackReviewer, MockStructuredReviewer, OpenRouterReviewer,
                         evaluate, run_checks_plus_reviewer, run_single_reviewer,
                         run_specialist_team)

PRICE_PER_MILLION_TOKENS = 0.15    # illustrative catalogue price, so you can redo the sums

def build_provider(scenario):
    """Live reviewer when a key exists (with a mock safety net), otherwise the mock."""
    if LIVE:
        return FallbackReviewer(OpenRouterReviewer(), MockStructuredReviewer(scenario))
    return MockStructuredReviewer(scenario)

def score_all_systems(scenario):
    """Run the three architectures with the same reviewer and return one row each."""
    provider = build_provider(scenario)
    runs = [run_single_reviewer(SOURCE, provider),
            run_checks_plus_reviewer(SOURCE, provider),
            run_specialist_team(SOURCE, provider)]
    return [evaluate(run, GOLDEN_PATH, price_per_million_tokens=PRICE_PER_MILLION_TOKENS)
            for run in runs]

COLUMNS = ["system", "found", "recall", "false_positives", "duplicates",
           "merged_duplicates", "model_calls", "tokens", "estimated_cost_usd"]

def cell_text(row, column):
    """Format one table cell; costs get fixed decimals so the column lines up."""
    if column == "estimated_cost_usd":
        return "%.6f" % row[column]
    return str(row[column])

def print_table(scenario, rows):
    print("Scenario:", scenario)
    widths = [max([len(col)] + [len(cell_text(row, col)) for row in rows]) for col in COLUMNS]
    print(" | ".join(col.ljust(w) for col, w in zip(COLUMNS, widths)))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(cell_text(row, col).ljust(w) for col, w in zip(COLUMNS, widths)))

print("Scoring function ready. Provider will be:",
      "live model with mock fallback" if LIVE else "MockStructuredReviewer")


## Step 3 — Scenario A: a reviewer with real blind spots

This is the case people have in mind when they reach for a team of agents.


In [ ]:
rows_a = score_all_systems("blind_spots")
print_table("blind_spots", rows_a)

print("\nWhat each system missed:")
for row in rows_a:
    print(f"   {row['system']:<22} {row['missed'] or 'nothing'}")


### Try it yourself

Now we swap in a reviewer that is already strong on its own — same three architectures, same artifact. Before running the next cell, write down which system you think will win, and what "win" should even mean here.


In [ ]:
# --- Worked solution ---
rows_b = score_all_systems("strong_generalist")
print_table("strong_generalist", rows_b)

print()
best_found = max(row["found"] for row in rows_b)
reaching_it = [row for row in rows_b if row["found"] == best_found]
cheapest = min(reaching_it, key=lambda row: row["model_calls"])

print("Most defects any system found :", best_found, "/ 9")
print("Systems that reached that     :", [row["system"] for row in reaching_it])
print("Smallest system that reached it:", cheapest["system"],
      f"({cheapest['model_calls']} call, {cheapest['tokens']} tokens, "
      f"${cheapest['estimated_cost_usd']:.6f})")
print()
print('"Win" is not "found the most". It is "met the quality bar with the least".')
print("Here the team found nothing the single reviewer had not already found,")
print("and charged three times as much to do it.")


## Step 4 — Put both scenarios side by side

Same code, same artifact, opposite verdict. Read the table and say which system you would deploy in each world.


In [ ]:
print(f"{'scenario':<20}{'system':<24}{'found':>6}{'calls':>7}{'tokens':>8}{'cost $':>10}")
print("-" * 75)
for scenario, rows in (("blind_spots", rows_a), ("strong_generalist", rows_b)):
    for row in rows:
        print(f"{scenario:<20}{row['system']:<24}{row['found']:>6}{row['model_calls']:>7}"
              f"{row['tokens']:>8}{row['estimated_cost_usd']:>10.6f}")
    print()

for scenario, rows in (("blind_spots", rows_a), ("strong_generalist", rows_b)):
    single = rows[0]
    team = rows[2]
    gain = team["found"] - single["found"]
    extra = team["tokens"] - single["tokens"]
    print(f"{scenario:<20} team found {gain:+d} defect(s) for {extra:+d} extra tokens "
          f"and {team['model_calls'] - single['model_calls']:+d} extra calls")


## Step 5 — When a multi-agent team is NOT worth it

We now have measured evidence rather than an opinion. Write these down; they are the deliverable of Day 4.


In [ ]:
reasons = [
    ("One reviewer already meets the bar",
     "strong_generalist: single 8/9 vs team 8/9 -> 0 defects gained for 3x the calls."),
    ("The extra recall costs more than it is worth",
     "Compare cost per extra defect against the cost of the defect escaping."),
    ("The sub-tasks are not truly independent",
     "If a branch needs another branch's output, fan-out becomes a fragile pipeline."),
    ("The failure is a capability gap, not an attention gap",
     "DEF-COR-02 was missed by the generalist AND by the correctness specialist: a "
     "narrower prompt cannot supply knowledge the reviewer never had."),
    ("Debugging burden grows with branches",
     "3 branches + a merge rule = 4 places a wrong report can come from, not 1."),
]

for index, (headline, evidence) in enumerate(reasons, 1):
    print(f"{index}. {headline}")
    print(f"   evidence: {evidence}")


## Step 6 — A reference table that works with no network

`data/captured_comparison.json` holds a saved run so this notebook still shows a table if OpenRouter is down or your key has expired. It was produced offline by the mock reviewer, so it is a record of the classroom result — not evidence about any real language model.


In [ ]:
import json

captured_path = PROJECT_ROOT / "data" / "captured_comparison.json"
captured = json.loads(captured_path.read_text(encoding="utf-8"))

print("Note from the file:", captured["note"])
print("Provider used     :", captured["provider"])

# Which table should your write-up cite? Whichever one actually ran.
if LIVE:
    print("Status            : you ran live, so cite YOUR tables above and use this")
    print("                    file only for the runs where a call failed.")
else:
    print("Status            : no API key, so this file and your tables agree by")
    print("                    construction - both came from the mock reviewer.")
print()

for scenario, rows in captured["scenarios"].items():
    print(scenario)
    for row in rows:
        print(f"   {row['system']:<24} found {row['found']}/9 | "
              f"calls {row['model_calls']} | tokens {row['tokens']}")


## Required live observation

Run one single-reviewer and one bounded specialist comparison with the issued model. Preserve raw structured results; use the captured comparison if the service is unavailable.


### Checkpoint

**1. In the `strong_generalist` scenario the specialist team still found 8 defects — as many as any other system. Why is that not a win?**

<details><summary>Show answer</summary>

Because it found nothing the single reviewer had not already found, while making 3 model calls instead of 1 and using roughly three times the tokens. It also produced duplicates that the supervisor had to merge, and three extra places for a bug to hide. The right question is never "which found the most" but "which is the smallest system that meets the bar".

</details>

**2. Both scenarios use exactly the same architecture code. So what actually changed the conclusion?**

<details><summary>Show answer</summary>

The reviewer's capability. `blind_spots` and `strong_generalist` differ only in which defects the reviewer can see. That is why "multi-agent is better" is never a general claim: it means "decomposition compensated for *this* reviewer on *this* task with *this* prompt", and it must be re-measured when any of those change.

</details>

### Recap

- Limitation we saw: a single comparison table can make a team of agents look unconditionally better than one reviewer.
- Layer we added: the same measurement repeated against a second reviewer, plus a captured offline table so the comparison always runs.
- Evidence it worked: 5/9 -> 9/9 in one scenario and 8/9 -> 8/9 at 3x the cost in the other, from identical architecture code.
